In [1]:
!pip install datasets pandas transformers

In [2]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("Riksarkivet/swener_1800", split="eval")
ds

/Users/aau/Desktop/swener_1800/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['text', 'entity_types', 'entity_start_chars', 'entity_end_chars', 'id', 'word_start_chars', 'word_end_chars'],
    num_rows: 170
})

In [ ]:
from collections import defaultdict
import pandas as pd

def evaluate(ds, preds, labels=None, merge_org=None, name=""):
    """
    ds     : eval split of Riksarkivet/swener_1800
    preds  : iterable of (doc_id, start, end, label); char offsets into the original text
    labels : labels to score, e.g. ["PER", "LOC", "ORG"] for ScandiNER.
             None = every label in gt (after ORG merge). Predictions outside this set are dropped.
    """
    gt = set()
    if merge_org is None:  
        merge_org = labels is not None and "ORG" in labels
    norm = lambda l: "ORG" if merge_org and l.startswith("ORG") else l
    for doc_id, types, starts, ends in zip(
        ds["id"], ds["entity_types"], ds["entity_start_chars"], ds["entity_end_chars"]
    ):
        for t, s, e in zip(types, starts, ends):
            gt.add((doc_id, s, e, norm(t)))

    pred = {(d, s, e, norm(l)) for d, s, e, l in preds}

    keep = set(labels) if labels else {x[3] for x in gt}
    gt = {x for x in gt if x[3] in keep}
    pred = {x for x in pred if x[3] in keep}

    def index(spans):
        d = defaultdict(list)
        for doc, s, e, lab in spans:
            d[doc].append((s, e, lab))
        return d

    gi, pi = index(gt), index(pred)

    def hit(item, idx):  # overlaps at least one span with the same label
        doc, s, e, lab = item
        return any(s < e2 and s2 < e and lab2 == lab for s2, e2, lab2 in idx[doc])

    def prf(tp_p, n_p, tp_r, n_g):
        p = tp_p / n_p if n_p else 0
        r = tp_r / n_g if n_g else 0
        f = 2 * p * r / (p + r) if p + r else 0
        return round(p, 3), round(r, 3), round(f, 3)

    rows = []
    for lab in sorted(keep) + ["micro"]:
        g = {x for x in gt if lab == "micro" or x[3] == lab}
        p = {x for x in pred if lab == "micro" or x[3] == lab}
        strict = prf(len(g & p), len(p), len(g & p), len(g))
        relaxed = prf(sum(hit(x, gi) for x in p), len(p),
                      sum(hit(x, pi) for x in g), len(g))
        rows.append((lab, len(g), len(p), *strict, *relaxed))

    res = pd.DataFrame(rows, columns=["label", "gt", "pred",
                                      "P_strict", "R_strict", "F1_strict",
                                      "P_relax", "R_relax", "F1_relax"])
    per_label = res[res.label != "micro"]
    print(f"== {name} | labels: {len(keep)} | "
          f"macro F1 strict {per_label.F1_strict.mean():.3f} / relaxed {per_label.F1_relax.mean():.3f}")
    display(res)
    return res

In [11]:
from transformers import pipeline

ner = pipeline(
    task="ner",
    model="saattrupdan/nbailab-base-ner-scandi",
    aggregation_strategy="first",
    framework="pt",
    device=0,
)
print("fast tokenizer:", ner.tokenizer.is_fast)


Device set to use mps:0


fast tokenizer: True


In [12]:
result = ner("Borghild kjøper seg inn i Bunnpris.")
pd.DataFrame.from_records(result)

,entity_group,score,word,start,end
0,PER,0.997718,Borghild,0,8
1,ORG,0.992670,Bunnpris,26,34


In [13]:
from tqdm.auto import tqdm

texts = ds["text"]
ids = ds["id"]

preds = []  # (doc_id, start, end, label)
for doc_id, text in tqdm(zip(ids, texts), total=len(texts)):
    for e in ner(text, stride=64):  # window over long text
        preds.append((doc_id, e["start"], e["end"], e["entity_group"]))

print(len(preds))
pd.DataFrame(preds, columns=["id", "start", "end", "label"]).label.value_counts()

100%|██████████| 170/170 [00:16<00:00, 10.54it/s]

2379


label
PER     1505
LOC      723
ORG       97
MISC      54
Name: count, dtype: int64

In [14]:
res_scandi = evaluate(ds, preds, labels=["PER", "LOC", "ORG"], name="ScandiNER PER/LOC/ORG")

== ScandiNER PER/LOC/ORG | labels: 3 | macro F1 strict 0.272 / relaxed 0.477


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,LOC,1350,723,0.562,0.301,0.392,0.842,0.436,0.575
1,ORG,455,97,0.268,0.057,0.094,0.619,0.103,0.177
2,PER,2488,1505,0.437,0.264,0.330,0.915,0.541,0.680
3,micro,4293,2325,0.469,0.254,0.329,0.880,0.462,0.606


In [ ]:
print(ner.tokenizer.model_max_length)


NameError: name 'ner' is not defined